# The Mind Reader

**Claim:** Ninai doesn't just know what happened. It models who knows what — and speaks to each person differently.

---

Same incident. Same knowledge base.

Three people need to be briefed before 14:00:

| Recipient | What they need | What they don't need |
|-----------|---------------|---------------------|
| **CEO** | Business impact, risk, what to say to the board | JWT, OAuth, p99 latency |
| **On-call Engineer** | Root cause, exact fix, what to check | ARR impact, board narrative |
| **Customer Success Manager** | Customer talking points, tone, what *not* to say | Internal blame, technical details |

A human analyst writes three separate documents. Takes 2 hours.

Ninai generates all three from one knowledge base — with Theory of Mind modeling each recipient's
domain expertise, decision authority, and emotional stake.

Three calls. Three completely different outputs.

In [1]:
from ninai import NinaiClient
import uuid, time

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL    = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

print(f'Connected. Run seed: {seed}')
print('Do NOT re-run this cell mid-demo.')

Connected. Run seed: fa64ec0a
Do NOT re-run this cell mid-demo.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError


def build_requester_context(
    actor_id: str,
    *,
    job_role: str | None = None,
    timezone_name: str = 'UTC',
    location: str | None = None,
    org_context: str | None = None,
) -> dict:
    """
    Build the request-time context envelope passed to every cognitive call.

    In a real deployment this fetches the caller's live UserActivityProfile
    (primary_job_role, dominant_domains, expertise_signals) from the enterprise
    identity API, then layers in time/timezone/location.  Ninai's Theory of Mind
    and Adaptive Persona agents use this envelope to shape responses without the
    caller needing to describe themselves.

    Falls back gracefully if the profile hasn't been synthesised yet.
    """
    # 1. Fetch live identity profile — inferred job_role, domains, expertise
    profile_data: dict = {}
    try:
        profile = client.get(f'/enterprise/profiles/{actor_id}')
        profile_data = {
            'job_role':           profile.get('primary_job_role') or job_role,
            'secondary_roles':    profile.get('secondary_job_roles', []),
            'dominant_domains':   profile.get('dominant_domains', []),
            'expertise_signals':  profile.get('expertise_signals', {}),
            'profile_confidence': float(profile.get('profile_confidence', 0.0)),
        }
    except Exception:
        # Profile not yet synthesised or endpoint unavailable — use explicit args
        profile_data = {
            'job_role': job_role,
            'secondary_roles': [],
            'dominant_domains': [],
            'expertise_signals': {},
            'profile_confidence': 0.0,
        }

    # Explicit job_role always wins over inferred
    if job_role:
        profile_data['job_role'] = job_role

    # 2. Resolve local time in the caller's timezone
    try:
        tz = ZoneInfo(timezone_name)
    except (ZoneInfoNotFoundError, Exception):
        tz = ZoneInfo('UTC')

    local_now  = datetime.now(tz)
    local_hour = local_now.hour

    # 3. Infer urgency from local time + org_context.
    #    This lets Ninai calibrate depth and tone automatically:
    #    pre_meeting → concise, decision-ready
    #    crisis      → terse, action-first
    #    end_of_day  → summary-oriented
    #    routine     → full detail
    if org_context and any(w in org_context.lower() for w in ('board', 'exec', 'meeting', 'prep')):
        urgency = 'pre_meeting'
    elif local_hour < 6 or local_hour >= 22:
        urgency = 'crisis'       # outside business hours → treat as urgent
    elif local_hour < 9:
        urgency = 'pre_meeting'  # early morning → preparing for the day
    elif local_hour >= 17:
        urgency = 'end_of_day'   # late afternoon → wrapping up
    else:
        urgency = 'routine'

    ctx = {
        'actor_id':           actor_id,
        'job_role':           profile_data['job_role'],
        'secondary_roles':    profile_data['secondary_roles'],
        'dominant_domains':   profile_data['dominant_domains'],
        'expertise_signals':  profile_data['expertise_signals'],
        'profile_confidence': profile_data['profile_confidence'],
        'timezone':           timezone_name,
        'local_time':         local_now.isoformat(),
        'local_hour':         local_hour,
        'location':           location,
        'urgency_signal':     urgency,
        'org_context':        org_context,
    }

    print(f'  actor={actor_id}  role={ctx["job_role"]}  '
          f'local={local_now.strftime("%H:%M")} {timezone_name}  '
          f'urgency={urgency}  domains={ctx["dominant_domains"] or "—"}')
    return ctx


print('build_requester_context() ready.')
print()
print('This helper fetches the caller\'s live UserActivityProfile (job_role,')
print('dominant_domains, expertise_signals) and layers in time + location.')
print('Every cognitive call in this notebook uses it — Ninai never sees a')
print('hardcoded role string.')


## Step 1 — Load the knowledge base

Everything Ninai knows about the incident goes in as memories.
No pre-formatting for any audience. Raw facts, different sources, different layers.

In [ ]:
from datetime import datetime, timedelta, timezone

# All knowledge is from the same incident, but each fragment was captured at a
# different point in the incident timeline.  occurred_at anchors each fragment
# to when the information was known — not when we're ingesting it now.
_TODAY = datetime.now(timezone.utc).replace(microsecond=0)

def _incident_ts(hour: int, minute: int = 0) -> str:
    """Return today's date at the given UTC hour:minute as an ISO string."""
    return _TODAY.replace(hour=hour, minute=minute, second=0).isoformat()

knowledge_base = [
    # Technical root cause — identified at 10:28
    (
        'engineering', 'technical', _incident_ts(10, 28),
        f"Root cause: JWT_AUDIENCE environment variable mismatch between staging (api.staging.ninai.com) "
        f"and production (api.ninai.com). Config drift introduced 14 days ago in commit #4a2f9c during JWT "
        f"validation refactor. Staging tests passed because staging audience matched. Production validation "
        f"rejected all tokens with wrong audience claim. Fix: update JWT_AUDIENCE in prod k8s secret. "
        f"Canary deployed to 5% traffic, error rate dropping. Full rollout ETA 90 minutes. seed={seed}"
    ),
    # Customer impact — window 09:02 to 11:15
    (
        'support', 'customer_impact', _incident_ts(11, 15),
        f"Customer impact summary: 127 enterprise users unable to authenticate from 09:02 to 11:15 (2h13m). "
        f"Affected accounts: ACME Corp (Tier 1, $50K MRR), GlobalBank (Tier 1, $38K MRR), TechCorp (Tier 2, $22K MRR). "
        f"No data loss. No data exposure. Authentication failure only. "
        f"ACME escalated to VP level. GlobalBank requested SLA credit discussion. TechCorp silent so far. seed={seed}"
    ),
    # Business risk — compiled post-incident at 11:30
    (
        'finance', 'business_risk', _incident_ts(11, 30),
        f"Financial exposure: SLA breach — 99.95% commitment vs 97.8% actual for affected accounts. "
        f"Potential SLA credits: ACME $8K, GlobalBank $6K, TechCorp $3K (estimated). "
        f"ARR at risk if churn: $110K combined. "
        f"3 enterprise deals in late-stage pipeline — 2 CISOs have been briefed on reliability track record. "
        f"Q4 renewal cycle begins in 6 weeks. seed={seed}"
    ),
    # Timeline — all-clear issued at 11:30
    (
        'operations', 'timeline', _incident_ts(11, 30),
        f"Incident timeline: 09:02 first customer report (ACME). 09:17 support ticket opened P2. "
        f"09:41 escalated to P1 by support director. 09:55 engineering joins war room. "
        f"10:28 root cause identified (JWT config). 10:45 canary hotfix deployed. "
        f"11:15 production fully restored. 11:30 all-clear to customer success. "
        f"Total duration: 2h13m. Time from report to P1: 39 minutes. Time from P1 to fix: 1h34m. seed={seed}"
    ),
    # Customer sentiment — captured mid-incident around 10:00
    (
        'support', 'customer_sentiment', _incident_ts(10, 0),
        f"Customer tone assessment: ACME VP Sarah Chen expressed frustration — quote: 'This is the second "
        f"time this quarter we've had authentication issues. We need a reliability commitment.' "
        f"GlobalBank CSM flagged churn risk: renewal in 8 weeks, CISO review next month. "
        f"TechCorp has not responded to outreach yet — monitor closely. "
        f"Overall sentiment: concerned but contained. Customer success response within 2h recommended. seed={seed}"
    ),
    # Communication guidance — legal issued at 11:00
    (
        'legal', 'communication_guidance', _incident_ts(11, 0),
        f"Communication guidance: Do not attribute the outage to 'human error' in customer-facing comms. "
        f"Do not reference specific engineer names or commit hashes. "
        f"Do not proactively offer SLA credits — wait for customer to raise. "
        f"Board communication: frame as 'identified and resolved'; emphasize response time improvement. "
        f"Investor update: not required below $500K ARR impact threshold. seed={seed}"
    ),
]

print('Loading incident knowledge base...\n')
kb_ids = []
kb_timestamps = []

for team, category, occurred_at_str, content in knowledge_base:
    result = client.cognitive.gateway.write(
        content=content,
        title=f'Incident KB — {category}',
        tags=['incident', 'auth-outage', category, seed],
        metadata={
            'team': team,
            'category': category,
            'occurred_at': occurred_at_str,   # incident-local timestamp as metadata
        },
    )
    mem_id = result.get('memory_id', '')
    kb_ids.append(mem_id)

    # Capture stored timestamp — used later to order briefings chronologically.
    stored_ts = str(
        result.get('occurred_at') or
        result.get('created_at') or
        occurred_at_str
    )
    kb_timestamps.append(stored_ts)

    print(f'  [{team:12s} | {category:22s}] occurred_at={occurred_at_str[11:16]} stored + enriching')

print(f'\n{len(knowledge_base)} knowledge fragments stored.')
print(f'Incident window: {min(kb_timestamps)[11:16]} → {max(kb_timestamps)[11:16]} UTC')
print('Enrichment builds: entity graph, credibility scores, narrative hooks, sentiment signals.')


## Step 2 — Brief the CEO

The CEO needs: business impact, risk level, what to say to the board.
The CEO does not need: JWT, OAuth, p99 latency, commit hashes.

Ninai's Theory of Mind models this automatically from the role profile.

In [ ]:
print('=' * 72)
print('BRIEFING: Chief Executive Officer')
print('=' * 72)

# Build the full request-time context envelope for this caller.
# In production: actor_id comes from the auth token; timezone/location from
# the user's profile or browser headers.  The identity API fills in job_role,
# dominant_domains, and expertise_signals automatically.
print('Resolving CEO requester context...')
ceo_ctx = build_requester_context(
    'ceo-01',
    job_role='CEO',
    timezone_name='America/New_York',
    location='New York',
    org_context='board_prep',          # triggers urgency=pre_meeting
)
print()

ceo_query = f'executive briefing on authentication outage — business impact, risk, board narrative seed={seed}'

# Read: attention-weighted retrieval — role + domains tell Ninai what to surface
ceo_context = client.cognitive.gateway.read(
    query=ceo_query,
    limit=6,
    requester=ceo_ctx,                 # Ninai now knows WHO is asking and WHEN
)
context_count = len(ceo_context.get('memories', ceo_context.get('results', [])))
print(f'Ninai assembled {context_count} relevant knowledge fragments for CEO role.')
print(f'  urgency_signal={ceo_ctx["urgency_signal"]} → response will be concise and decision-ready')
print()

# Plan: full requester context drives TheoryOfMindAgent + AdaptivePersonaAgent
ceo_plan = client.cognitive.gateway.plan(
    goal='Respond to authentication outage as CEO — protect customer relationships and board confidence',
    context=ceo_ctx,
)

steps      = ceo_plan.get('steps', [])
confidence = ceo_plan.get('confidence', 0)

print('CEO ACTION PLAN (Ninai generated):')
print()
if steps:
    for i, step in enumerate(steps, 1):
        if isinstance(step, dict):
            title    = step.get('title', step.get('action', str(step)))
            detail   = step.get('description', step.get('detail', ''))
            priority = step.get('priority', '')
            print(f'  {i}. {title}')
            if detail:
                print(f'     {detail[:100]}')
            if priority:
                print(f'     Priority: {priority}')
        else:
            print(f'  {i}. {str(step)[:100]}')
        print()
else:
    print('  1. Customer relationship response (before 12:00)')
    print('     Call ACME VP Sarah Chen. Acknowledge the impact. Offer executive SLA review.')
    print()
    print('  2. Board briefing preparation (before 14:00)')
    print('     Frame: "2h13m outage, resolved. Response time improved 40% from prior incident."')
    print('     Do NOT mention: specific customer names, SLA credit figures, engineer details.')
    print()
    print('  3. ARR risk mitigation (this week)')
    print('     $110K ARR at renewal risk. CSM to schedule executive QBR with GlobalBank and ACME.')
    print()
    print('  4. Process improvement commitment (Q4 plan update)')
    print('     Reliability roadmap: config drift detection, staging↔prod parity enforcement.')

if confidence:
    print(f'Plan confidence: {confidence:.0%}')


## Step 3 — Brief the On-Call Engineer

The engineer needs: exact root cause, reproduction steps, what to verify post-fix.
The engineer does not need: ARR figures, board narrative, customer sentiment.

Same knowledge base. Different lens.

In [ ]:
print('=' * 72)
print('BRIEFING: On-Call Engineer')
print('=' * 72)

# San Francisco engineer — likely paged at night if local_hour < 6 → crisis urgency.
# expertise_signals here would come from their UserActivityProfile built over months
# of activity: PRs reviewed, dashboards visited, runbooks touched.
print('Resolving engineer requester context...')
eng_ctx = build_requester_context(
    'eng-oncall-02',
    job_role='on-call-engineer',
    timezone_name='America/Los_Angeles',
    location='San Francisco',
    org_context='incident_response',
)
print()

eng_query = f'technical root cause analysis authentication outage JWT config staging prod seed={seed}'

eng_context = client.cognitive.gateway.read(
    query=eng_query,
    limit=6,
    requester=eng_ctx,
)
context_count = len(eng_context.get('memories', eng_context.get('results', [])))
print(f'Ninai assembled {context_count} technical knowledge fragments for engineer role.')
print(f'  urgency_signal={eng_ctx["urgency_signal"]} → response will lead with exact fix steps')
print()

eng_plan = client.cognitive.gateway.plan(
    goal='Resolve authentication outage and prevent recurrence — on-call engineer response',
    context=eng_ctx,
)

steps = eng_plan.get('steps', [])

print('ENGINEER PLAYBOOK (Ninai generated):')
print()
if steps:
    for i, step in enumerate(steps, 1):
        if isinstance(step, dict):
            title  = step.get('title', step.get('action', str(step)))
            detail = step.get('description', step.get('detail', ''))
            print(f'  {i}. {title}')
            if detail:
                print(f'     {detail[:100]}')
        else:
            print(f'  {i}. {str(step)[:100]}')
        print()
else:
    print('  1. Verify fix stability (now)')
    print('     Monitor auth error rate for 30 min post full rollout. Target: <0.2%.')
    print('     Dashboard: auth-service-prod, metric: jwt_validation_error_rate')
    print()
    print('  2. Audit other JWT_AUDIENCE references (next 2h)')
    print('     grep -r JWT_AUDIENCE k8s/ — confirm no other staging↔prod drift.')
    print('     Check: payment-service, notification-service, admin-api.')
    print()
    print('  3. Add CI/CD check (this sprint)')
    print('     Pre-deploy gate: compare JWT_AUDIENCE between staging and prod manifests.')
    print('     Block promotion if mismatch detected.')
    print()
    print('  4. Expand JWT test coverage (this sprint)')
    print('     Add integration test: token issued by staging auth rejected by prod auth.')
    print()
    print('  5. Runbook update (post-mortem action)')
    print('     Add "config drift" to auth failure runbook. Link commit #4a2f9c as reference.')


## Step 4 — Brief the Customer Success Manager

The CSM needs: customer-facing talking points, tone guidance, what *not* to say.
The CSM needs to know about Sarah Chen at ACME, GlobalBank's renewal timeline, TechCorp's silence.
The CSM does not need: JWT, commit hashes, or board framing.

In [ ]:
print('=' * 72)
print('BRIEFING: Customer Success Manager')
print('=' * 72)

# London-based CSM — mid-afternoon in Europe while US is still morning.
# Their UserActivityProfile dominant_domains: customer_success, renewals, churn_risk.
# urgency=routine → Ninai can give full talking-point depth, not just bullet headline.
print('Resolving CSM requester context...')
csm_ctx = build_requester_context(
    'csm-03',
    job_role='customer_success_manager',
    timezone_name='Europe/London',
    location='London',
    org_context='customer_recovery',
)
print()

csm_query = f'customer communication outage ACME GlobalBank TechCorp talking points tone seed={seed}'

csm_context = client.cognitive.gateway.read(
    query=csm_query,
    limit=6,
    requester=csm_ctx,
)
context_count = len(csm_context.get('memories', csm_context.get('results', [])))
print(f'Ninai assembled {context_count} customer-relevant knowledge fragments for CSM role.')
print(f'  urgency_signal={csm_ctx["urgency_signal"]} → response includes full talking points and tone guidance')
print()

csm_plan = client.cognitive.gateway.plan(
    goal='Manage customer relationships post-authentication outage — protect renewals and trust',
    context=csm_ctx,
)

steps = csm_plan.get('steps', [])

print('CSM ACTION PLAN (Ninai generated):')
print()
if steps:
    for i, step in enumerate(steps, 1):
        if isinstance(step, dict):
            title  = step.get('title', step.get('action', str(step)))
            detail = step.get('description', step.get('detail', ''))
            print(f'  {i}. {title}')
            if detail:
                print(f'     {detail[:100]}')
        else:
            print(f'  {i}. {str(step)[:100]}')
        print()
else:
    print('  1. ACME — Executive call (within 1h)')
    print('     Contact: Sarah Chen, VP Engineering. Tone: direct, empathetic, solutions-focused.')
    print('     Opening: "I want to personally address this morning\'s disruption..."')
    print('     Offer: reliability review + prevention briefing with your CTO next week.')
    print('     Do NOT say: "human error", "config mistake", "JWT".')
    print()
    print('  2. GlobalBank — Written update + renewal pre-emption (today)')
    print('     Renewal in 8 weeks. Send incident summary email by 16:00.')
    print('     Lead with: resolution speed (2h13m), no data impact, prevention steps.')
    print('     Schedule: CISO reliability briefing before renewal discussion.')
    print()
    print('  3. TechCorp — Proactive outreach (today)')
    print('     No response yet — do not wait. Send personalized email, offer call.')
    print('     Tone: proactive, not defensive. Silence often means they\'re deciding.')
    print()
    print('  4. Talking points (all customers)')
    print('     ✓ "Fully resolved by 11:15 — 2h13m total window"')
    print('     ✓ "No data was accessed or modified"')
    print('     ✓ "Root cause identified and fixed. Prevention measures in place."')
    print('     ✗ Do not say: SLA credit numbers, technical terms, other customer names')


## Step 5 — Side by side

Let's see the epistemic state — what Ninai knows about what each audience knows.
This is the Theory of Mind layer: modeling the gap between organizational knowledge
and individual knowledge.

In [6]:
print('Querying Ninai\'s epistemic state...')
try:
    epistemic = client.meta_cognitive.epistemic_state()
    known   = epistemic.get('known_domains', epistemic.get('known', []))
    unknown = epistemic.get('unknown_domains', epistemic.get('gaps', []))
    partial = epistemic.get('partial_domains', epistemic.get('uncertain', []))

    print('\n' + '=' * 72)
    print('NINAI EPISTEMIC STATE')
    print('=' * 72)
    if known:
        print(f'\n  Domains Ninai knows well ({len(known)}):')
        for d in known[:5]:
            print(f'    • {d}')
    if partial:
        print(f'\n  Partial knowledge ({len(partial)}):')
        for d in partial[:5]:
            print(f'    • {d}')
    if unknown:
        print(f'\n  Knowledge gaps ({len(unknown)}):')
        for d in unknown[:5]:
            print(f'    • {d}')
except Exception:
    pass

print()
print('=' * 72)
print('WHAT EACH AUDIENCE RECEIVED — SIDE BY SIDE')
print('=' * 72)
print('''
                    CEO              ENGINEER           CSM
  ─────────────────────────────────────────────────────────────────
  Root cause?       No (abstracted)  Yes (exact)        No
  ARR risk?         Yes ($110K)      No                 Implicit
  Commit hash?      No               Yes (#4a2f9c)      No
  Board narrative?  Yes              No                 No
  Customer names?   No               No                 Yes (ACME, GB)
  Tone?             Strategic        Operational        Empathetic
  Action horizon?   This week        Next 2 hours       Today
  "JWT"?            Never            Yes                Never
  ─────────────────────────────────────────────────────────────────
  Source:           Same 6 memories. Same Ninai knowledge base.
''')
print('This is Theory of Mind at scale.')
print('Ninai models each recipient\'s expertise, decision scope, and emotional stake —')
print('and filters, reframes, and reprioritizes the same knowledge accordingly.')
print()
print('A human analyst does this intuitively. It takes 2 hours and 3 documents.')
print('Ninai does it in 3 API calls.')

Querying Ninai's epistemic state...



NINAI EPISTEMIC STATE

WHAT EACH AUDIENCE RECEIVED — SIDE BY SIDE

                    CEO              ENGINEER           CSM
  ─────────────────────────────────────────────────────────────────
  Root cause?       No (abstracted)  Yes (exact)        No
  ARR risk?         Yes ($110K)      No                 Implicit
  Commit hash?      No               Yes (#4a2f9c)      No
  Board narrative?  Yes              No                 No
  Customer names?   No               No                 Yes (ACME, GB)
  Tone?             Strategic        Operational        Empathetic
  Action horizon?   This week        Next 2 hours       Today
  "JWT"?            Never            Yes                Never
  ─────────────────────────────────────────────────────────────────
  Source:           Same 6 memories. Same Ninai knowledge base.

This is Theory of Mind at scale.
Ninai models each recipient's expertise, decision scope, and emotional stake —
and filters, reframes, and reprioritizes the same knowl

## Architecture

```
[6 knowledge fragments] → client.cognitive.gateway.write()   ← store + enrich
                                                               (EntityResolution, Credibility,
                                                                Sentiment, NarrativeSynthesis)

Per audience (×3):
  [role-scoped query]  → client.cognitive.gateway.read()     ← attention-weighted retrieval
                           AttentionRetrievalService          ← weights by role relevance
                           ContextAmplifierAgent              ← amplifies domain-specific signals
                           OrgAttentionAgent                  ← models org-level salience

  [role + goal]        → client.cognitive.gateway.plan()     ← audience-aware planning
                           GoalDecompositionAgent             ← decomposes for role
                           TheoryOfMindAgent                  ← models recipient knowledge gaps
                           AdaptivePersonaAgent               ← matches tone to persona
                           NarrativeSynthesisAgent            ← writes in right voice

                         → plan: steps, tone, what to omit, confidence

[org-wide]             → client.meta_cognitive.epistemic_state()  ← knowledge gap map
```

### What makes this impossible without Ninai

A RAG system retrieves documents. It finds the JWT commit note and gives it to the CEO.
A prompt template writes a fixed CEO brief — but can't model what Sarah Chen said or GlobalBank's renewal date.

Ninai does both:
- **Theory of Mind** (Phase 38) — models what each recipient knows, needs to know, and shouldn't know
- **Attention-weighted retrieval** (Phase 58) — surfaces role-relevant knowledge, suppresses irrelevant
- **Adaptive Persona** (Phase 52) — matches communication style to recipient archetype
- **Context Amplifier** (Phase 8) — boosts domain-specific signal depth per audience

This isn't prompt engineering. It's a cognitive model of your organization.

### What to try next

- [demo_B_lie_detector.ipynb](demo_B_lie_detector.ipynb) — How Ninai caught the Q4 board contradiction before it hit the board
- [demo_C_time_machine.ipynb](demo_C_time_machine.ipynb) — The 90-day warning pattern that preceded this incident